# 🏥 PharmaAgent - Backend Server (Google Colab)

This notebook runs the PharmaAgent FastAPI backend on Google Colab and exposes it publicly via **ngrok**.

## Setup Steps:
1. **Cell 1**: Upload project files
2. **Cell 2**: Install dependencies
3. **Cell 3**: Configure ngrok
4. **Cell 4**: Start the server

> ⚠️ **Note**: Ollama is NOT available on Colab, so the backend runs in **rule-based mode** (no LLM). All core features still work.

---

## Cell 1: Clone or Upload Project

**Option A**: Clone from GitHub (recommended if your repo is public/private with token)

**Option B**: Upload manually using the file browser on the left

In [2]:
# ============================================================
# OPTION A: Clone from GitHub
# Replace with your actual GitHub repo URL
# ============================================================

# For public repos:
# !git clone https://github.com/YOUR_USERNAME/pharma-agent.git

# For private repos (use personal access token):
# !git clone https://YOUR_TOKEN@github.com/YOUR_USERNAME/pharma-agent.git

# ============================================================
# OPTION B: Upload from Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Copy from Drive (adjust path as needed)
# !cp -r "/content/drive/MyDrive/pharma-agent" /content/pharma-agent

# ============================================================
# OPTION C: Upload a zip file
# ============================================================
# from google.colab import files
# uploaded = files.upload()  # Upload pharma-agent.zip
# !unzip pharma-agent.zip -d /content/

print("✅ Choose one option above, uncomment it, and run this cell")

ModuleNotFoundError: No module named 'google'

## Cell 2: Install Dependencies

In [3]:
# ============================================================
# Install Python packages
# ============================================================
import os

# Set the project root - adjust this path if your folder has a different name
PROJECT_ROOT = "/content/pharma-agent"
BACKEND_DIR = os.path.join(PROJECT_ROOT, "backend")

# Verify the directory exists
assert os.path.exists(BACKEND_DIR), f"❌ Backend directory not found at {BACKEND_DIR}. Check Cell 1."
print(f"📁 Backend directory: {BACKEND_DIR}")

# Install requirements
!pip install -q -r {BACKEND_DIR}/requirements.txt

# Install ngrok
!pip install -q pyngrok

# Create uploads directory
os.makedirs(os.path.join(BACKEND_DIR, "uploads/prescriptions"), exist_ok=True)

print("\n✅ All dependencies installed!")

AssertionError: ❌ Backend directory not found at /content/pharma-agent/backend. Check Cell 1.

## Cell 3: Configure ngrok

Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken

In [4]:
# ============================================================
# Configure ngrok auth token
# ============================================================
from getpass import getpass
from pyngrok import ngrok, conf

# Enter your ngrok auth token (get it from https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = getpass("🔑 Enter your ngrok auth token: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok configured!")

# Optional: If you have a paid ngrok plan with a static domain, set it here
# NGROK_STATIC_DOMAIN = "your-domain.ngrok-free.app"  # Uncomment and set if you have one

ModuleNotFoundError: No module named 'pyngrok'

## Cell 4: Start FastAPI Server + ngrok Tunnel

This cell starts the backend and creates a public tunnel. **Keep this cell running!**

In [5]:
# ============================================================
# Start the FastAPI backend with ngrok tunnel
# ============================================================
import subprocess
import sys
import os
import time
import threading
from pyngrok import ngrok

PROJECT_ROOT = "/content/pharma-agent"
BACKEND_DIR = os.path.join(PROJECT_ROOT, "backend")
PORT = 8000

# Add project root to Python path (for agent imports)
sys.path.insert(0, PROJECT_ROOT)
os.chdir(BACKEND_DIR)

# Kill any existing ngrok tunnels
ngrok.kill()

# Start uvicorn in a background thread
def run_server():
    subprocess.run([
        sys.executable, "-m", "uvicorn",
        "app.main:app",
        "--host", "0.0.0.0",
        "--port", str(PORT)
    ], cwd=BACKEND_DIR)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
print("⏳ Starting FastAPI server...")
time.sleep(5)

# Create ngrok tunnel
# If you have a static domain, use:
# public_url = ngrok.connect(PORT, domain="your-domain.ngrok-free.app")
public_url = ngrok.connect(PORT)

print("\n" + "=" * 60)
print("🚀 PharmaAgent Backend is LIVE!")
print("=" * 60)
print(f"\n🌐 Public URL:  {public_url}")
print(f"📖 API Docs:    {public_url}/docs")
print(f"❤️  Health:      {public_url}/health")
print(f"\n📋 Copy this URL and paste it as VITE_API_URL in Vercel:")
print(f"\n   👉  {public_url}")
print("\n" + "=" * 60)
print("⚠️  Keep this cell running! Stopping it kills the server.")
print("=" * 60)

# Keep alive - this cell will run indefinitely
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n👋 Server stopped.")
    ngrok.kill()

ModuleNotFoundError: No module named 'pyngrok'

## Cell 5 (Optional): Test the API

Run this in a **separate cell** while Cell 4 is running to verify the API works.

In [6]:
# ============================================================
# Quick API test (run while Cell 4 is active)
# ============================================================
import requests

# Get the active tunnel URL
from pyngrok import ngrok
tunnels = ngrok.get_tunnels()
if tunnels:
    base_url = tunnels[0].public_url
    print(f"Testing: {base_url}")

    # Health check
    r = requests.get(f"{base_url}/health")
    print(f"\n✅ Health: {r.json()}")

    # Chat test
    r = requests.post(f"{base_url}/chat",
                      json={"message": "hello"})
    print(f"\n✅ Chat response: {r.json()['response'][:200]}...")

    # Medicines
    r = requests.get(f"{base_url}/medicines")
    print(f"\n✅ Medicines: {len(r.json())} items loaded")
else:
    print("❌ No active tunnels. Run Cell 4 first.")

ModuleNotFoundError: No module named 'requests'

---

## Next Steps

1. Copy the **public URL** from Cell 4 output
2. Go to **Vercel** → Your project → **Settings** → **Environment Variables**
3. Set `VITE_API_URL` = the ngrok URL (e.g. `https://abc123.ngrok-free.app`)
4. **Redeploy** on Vercel
5. Your frontend will now connect to this Colab backend!

> **Tip**: With ngrok free tier, the URL changes each restart. With a paid plan ($8/mo), you get a static domain.